# Refinamento Camada Silver
Nesse notebook vamos verificar se as dimensões estão atuando conforme esperado, ou se ocorre alguma perda de dados

In [0]:
vra = spark.table("mvp_engenharia_dados.silver.silver_vra")
empresas = spark.table("mvp_engenharia_dados.silver.silver_empresas")
aerodromos = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

total_vra = vra.count()

print(f"=" * 60)
print(f"VERIFICAÇÃO DE INTEGRIDADE REFERENCIAL")
print(f"=" * 60)
print(f"Tabela fato (silver_vra): {total_vra:,d} registros")
print(f"Dimensão empresas: {empresas.count()} registros")
print(f"Dimensão aerodromos: {aerodromos.count()} registros")
print()

## Matches e misses por dimensão

Cada dimensão é testada individualmente via LEFT JOIN contra a tabela fato. O objetivo é medir quantos registros da VRA encontram correspondência e quantos ficam órfãos (miss) por falta de registro na dimensão.

In [0]:
from pyspark.sql.functions import col, count as count_col

vra_emp = vra.join(empresas, vra.sigla_icao_empresa_aerea == empresas.icao, "left")
emp_match = vra_emp.filter(col("icao").isNotNull()).count()
emp_miss = vra_emp.filter(col("icao").isNull()).count()

vra_aero_orig = vra.join(aerodromos, vra.sigla_icao_aeroporto_origem == aerodromos.codigo_oaci, "left")
aero_orig_match = vra_aero_orig.filter(col("codigo_oaci").isNotNull()).count()
aero_orig_miss = vra_aero_orig.filter(col("codigo_oaci").isNull()).count()

vra_aero_dest = vra.join(aerodromos, vra.sigla_icao_aeroporto_destino == aerodromos.codigo_oaci, "left")
aero_dest_match = vra_aero_dest.filter(col("codigo_oaci").isNotNull()).count()
aero_dest_miss = vra_aero_dest.filter(col("codigo_oaci").isNull()).count()

print("=" * 60)
print("RESULTADO POR DIMENSÃO (LEFT JOIN)")
print("=" * 60)
print(f"\n--- Empresas (sigla_icao_empresa_aerea → icao) ---")
print(f"  Match: {emp_match:,d} ({emp_match/total_vra*100:.2f}%)")
print(f"  Miss:  {emp_miss:,d} ({emp_miss/total_vra*100:.2f}%)")
print(f"\n--- Aerodromos Origem (sigla_icao_aeroporto_origem → codigo_oaci) ---")
print(f"  Match: {aero_orig_match:,d} ({aero_orig_match/total_vra*100:.2f}%)")
print(f"  Miss:  {aero_orig_miss:,d} ({aero_orig_miss/total_vra*100:.2f}%)")
print(f"\n--- Aerodromos Destino (sigla_icao_aeroporto_destino → codigo_oaci) ---")
print(f"  Match: {aero_dest_match:,d} ({aero_dest_match/total_vra*100:.2f}%)")
print(f"  Miss:  {aero_dest_miss:,d} ({aero_dest_miss/total_vra*100:.2f}%)")

## Detalhamento das chaves sem match

Para cada dimensão, agrupamos as chaves da VRA que não encontraram correspondência e contamos quantos registros são afetados por cada chave ausente. Isso permite priorizar quais registros faltantes na dimensão teriam maior impacto.

In [0]:
from pyspark.sql.functions import col, count as count_col

emp_keys_miss = (vra
    .join(empresas, vra.sigla_icao_empresa_aerea == empresas.icao, "left")
    .filter(col("icao").isNull())
    .groupBy("sigla_icao_empresa_aerea")
    .agg(count_col("*").alias("registros_afetados"))
    .orderBy(col("registros_afetados").desc()))

print("=" * 60)
print("CHAVES DE EMPRESA SEM MATCH NA DIMENSÃO")
print("=" * 60)
if emp_keys_miss.count() > 0:
    display(emp_keys_miss)
else:
    print("Nenhuma chave sem match!")

aero_orig_keys_miss = (vra
    .join(aerodromos, vra.sigla_icao_aeroporto_origem == aerodromos.codigo_oaci, "left")
    .filter(col("codigo_oaci").isNull())
    .groupBy("sigla_icao_aeroporto_origem")
    .agg(count_col("*").alias("registros_afetados"))
    .orderBy(col("registros_afetados").desc()))

print("\n" + "=" * 60)
print("CHAVES DE AERÓDROMO ORIGEM SEM MATCH NA DIMENSÃO")
print("=" * 60)
if aero_orig_keys_miss.count() > 0:
    display(aero_orig_keys_miss)
else:
    print("Nenhuma chave sem match!")

aero_dest_keys_miss = (vra
    .join(aerodromos, vra.sigla_icao_aeroporto_destino == aerodromos.codigo_oaci, "left")
    .filter(col("codigo_oaci").isNull())
    .groupBy("sigla_icao_aeroporto_destino")
    .agg(count_col("*").alias("registros_afetados"))
    .orderBy(col("registros_afetados").desc()))

print("\n" + "=" * 60)
print("CHAVES DE AERÓDROMO DESTINO SEM MATCH NA DIMENSÃO")
print("=" * 60)
if aero_dest_keys_miss.count() > 0:
    display(aero_dest_keys_miss)
else:
    print("Nenhuma chave sem match!")

## Impacto combinado do INNER JOIN

Simula o cenário real de uso: INNER JOIN com as três dimensões simultaneamente (empresas + aeródromos de origem e destino). O resultado mostra quantos registros da VRA sobreviveriam e quantos seriam perdidos no cruzamento completo.

In [0]:
vra_completo = (vra
    .join(empresas, vra.sigla_icao_empresa_aerea == empresas.icao, "inner")
    .join(aerodromos, vra.sigla_icao_aeroporto_origem == aerodromos.codigo_oaci, "inner")
    .join(aerodromos.alias("aero_dest"), vra.sigla_icao_aeroporto_destino == col("aero_dest.codigo_oaci"), "inner"))

total_inner = vra_completo.count()
perda = total_vra - total_inner
pct_perda = perda / total_vra * 100

print("=" * 60)
print("IMPACTO COMBINADO (INNER JOIN com as 3 dimensões)")
print("=" * 60)
print(f"Registros originais (VRA): {total_vra:,d}")
print(f"Registros ap\u00f3s INNER JOIN:   {total_inner:,d}")
print(f"Registros perdidos:        {perda:,d} ({pct_perda:.2f}%)")
print(f"Registros mantidos:        {total_inner/total_vra*100:.2f}%")

## Complementação com dados do SIROS/ANAC

Vamos baixar os CSVs oficiais do SIROS (aerodromos e cias) diretamente para o volume de staging e verificar se eles contêm os aeródromos e empresas que estão faltando na nossa dimensão silver.


In [0]:
import urllib.request
import os

volume_path = "/Volumes/mvp_engenharia_dados/staging/pontualidade"

urls = {
    "aerodromos_siros.csv": "https://siros.anac.gov.br/siros/registros/aerodromo/aerodromos.csv",
    "cias_siros.csv": "https://siros.anac.gov.br/siros/registros/cia/cias.csv",
}

for filename, url in urls.items():
    dest = f"{volume_path}/{filename}"
    print(f"Baixando {url} -> {dest}")
    urllib.request.urlretrieve(url, dest)
    size = os.path.getsize(dest)
    print(f"  OK: {size:,d} bytes")
    with open(dest, "r", encoding="utf-8", errors="replace") as f:
        header = f.readline().strip()
    print(f"  Header: {header}")
    print()

### Verificação de aeródromos faltantes

Ler o CSV de aeródromos baixado do SIROS e verificar se contém os códigos OACI que estão ausentes da nossa dimensão `silver_aerodromos`.

In [0]:
aero_siros = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/aerodromos_siros.csv"))

print(f"Total de aeródromos no SIROS: {aero_siros.count()}")
print(f"Colunas: {aero_siros.columns}")
display(aero_siros.limit(5))

In [0]:
missing_aero_orig = [r[0] for r in aero_orig_keys_miss.select("sigla_icao_aeroporto_origem").collect()]
missing_aero_dest = [r[0] for r in aero_dest_keys_miss.select("sigla_icao_aeroporto_destino").collect()]
missing_aero_all = list(set(missing_aero_orig + missing_aero_dest))

oaci_col = [c for c in aero_siros.columns if "oaci" in c.lower() or "icao" in c.lower() or "codigo" in c.lower()]
print(f"Colunas candidatas a código OACI: {oaci_col}")

if oaci_col:
    col_name = oaci_col[0]
    siros_codes = set([r[0] for r in aero_siros.select(col_name).distinct().collect()])
    
    print(f"\nVerificando {len(missing_aero_all)} códigos ausentes:")
    print(f"{'Código':<10} {'No SIROS?':<12} {'Registros VRA (orig+dest)'}")
    print("-" * 50)
    for code in sorted(missing_aero_all):
        in_siros = "SIM" if code in siros_codes else "NÃO"
        n_orig = sum(1 for c in missing_aero_orig if c == code)
        n_dest = sum(1 for c in missing_aero_dest if c == code)
        print(f"{code:<10} {in_siros:<12} {n_orig}+{n_dest}")
    
    found_in_siros = [c for c in missing_aero_all if c in siros_codes]
    not_in_siros = [c for c in missing_aero_all if c not in siros_codes]
    print(f"\nEncontrados no SIROS: {len(found_in_siros)}")
    print(f"Não encontrados no SIROS: {len(not_in_siros)} -> {not_in_siros}")
else:
    print("Não foi possível identificar a coluna de código OACI no CSV")

### Verificação de empresas faltantes

Ler o CSV de companhias aéreas baixado do SIROS e verificar se contém os códigos ICAO que estão ausentes da nossa dimensão `silver_empresas`.

In [0]:
cias_siros = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{volume_path}/cias_siros.csv"))

print(f"Total de empresas no SIROS: {cias_siros.count()}")
print(f"Colunas: {cias_siros.columns}")
display(cias_siros.limit(5))

In [0]:
missing_emp = [r[0] for r in emp_keys_miss.select("sigla_icao_empresa_aerea").collect()]

icao_col = [c for c in cias_siros.columns if "icao" in c.lower() or "icao" in c.lower() or "designador" in c.lower()]
print(f"Colunas candidatas a ICAO: {icao_col}")

if icao_col:
    col_name = icao_col[0]
    siros_icao = set([r[0] for r in cias_siros.select(col_name).distinct().collect() if r[0]])
    
    print(f"\nVerificando {len(missing_emp)} códigos ausentes:")
    print(f"{'Código':<10} {'No SIROS?':<12} {'Registros VRA'}")
    print("-" * 40)
    emp_counts = {r[0]: r[1] for r in emp_keys_miss.collect()}
    for code in sorted(missing_emp):
        in_siros = "SIM" if code in siros_icao else "NÃO"
        print(f"{code:<10} {in_siros:<12} {emp_counts.get(code, 0):,d}")
    
    found = [c for c in missing_emp if c in siros_icao]
    not_found = [c for c in missing_emp if c not in siros_icao]
    print(f"\nEncontrados no SIROS: {len(found)} -> {found}")
    print(f"Não encontrados no SIROS: {len(not_found)} -> {not_found}")
else:
    print("Não foi possível identificar a coluna de ICAO no CSV")

### Investigacão das empresas PTB e PAM

O SIROS não encontrou os códigos PTB, PAM e ARN. Vamos investigar:
- **PTB** = Voepass Linhas Aéreas (ex-Passaredo)
- **PAM** = MAP Linhas Aéreas
- **ARN** = apenas 1 registro, impacto irrelevante

Primeiro verificamos se essas empresas existem no SIROS sob outro código ICAO.

In [0]:
from pyspark.sql.functions import upper, col

display(cias_siros.filter(
    upper(col("NOME OPERADOR AÉREO")).contains("VOEPASS") |
    upper(col("NOME OPERADOR AÉREO")).contains("PASSAREDO") |
    upper(col("NOME OPERADOR AÉREO")).contains("MAP ") |
    upper(col("NOME OPERADOR AÉREO")).contains("MAP LINHAS")
))

> **Sobre o código ARN:** Verificado na lista oficial de códigos de companhias aéreas da Wikipedia ([List of airline codes](https://en.wikipedia.org/wiki/List_of_airline_codes)), o código ARN não consta. Com apenas 1 registro na VRA, trata-se de um valor isolado sem impacto relevante, a perda de 0,00% é desprezível.

### Inserir PTB e PAM manualmente

Como essas empresas não constam no SIROS nem na dimensão silver, mas existem 38.778 registros na VRA associados a elas, vamos inseri-las manualmente para recuperar esses dados.

| ICAO | Razão Social | Cidade | UF |
| --- | --- | --- | --- |
| PTB | VOEPASS LINHAS AÉREAS | RIBEIRÃO PRETO | SP |
| PAM | MAP LINHAS AÉREAS | SÃO PAULO | SP |

In [0]:
from pyspark.sql.functions import lit, concat, split

novas_empresas = spark.createDataFrame(
    [
        ("ATIVA", "RIBEIRÃO PRETO", "-", "PTB", "VOEPASS LINHAS AÉREAS", "SP", "VOEPASS (PTB)"),
        ("ATIVA", "SÃO PAULO", "-", "PAM", "MAP LINHAS AÉREAS", "SP", "MAP (PAM)"),
    ],
    schema="ativa string, cidade string, estrangeira string, icao string, razao string, uf string, empresa string"
)

empresas_atualizado = empresas.union(novas_empresas)

(empresas_atualizado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_empresas"))

print(f"silver_empresas atualizada: {empresas_atualizado.count()} empresas (antes: {empresas.count()})")
display(spark.table("mvp_engenharia_dados.silver.silver_empresas").filter(col("icao").isin("PTB", "PAM")))

### Re-executar verificacão de impacto

Com as duas empresas inseridas, re-executamos o INNER JOIN com as três dimensões para medir a nova taxa de perda.

In [0]:
empresas_nova = spark.table("mvp_engenharia_dados.silver.silver_empresas")

vra_emp_nova = vra.join(empresas_nova, vra.sigla_icao_empresa_aerea == empresas_nova.icao, "left")
emp_miss_nova = vra_emp_nova.filter(col("icao").isNull()).count()

vra_completo_novo = (vra
    .join(empresas_nova, vra.sigla_icao_empresa_aerea == empresas_nova.icao, "inner")
    .join(aerodromos, vra.sigla_icao_aeroporto_origem == aerodromos.codigo_oaci, "inner")
    .join(aerodromos.alias("aero_dest"), vra.sigla_icao_aeroporto_destino == col("aero_dest.codigo_oaci"), "inner"))

total_inner_novo = vra_completo_novo.count()
perda_nova = total_vra - total_inner_novo
pct_perda_nova = perda_nova / total_vra * 100

print("=" * 60)
print("VERIFICACÃO APÓS INSERÇÃO DE PTB E PAM")
print("=" * 60)
print(f"Empresas (miss): {emp_miss_nova} (antes: {emp_miss})")
print(f"Registros recuperados (empresas): {emp_miss - emp_miss_nova:,d}")
print()
print(f"Registros originais (VRA):      {total_vra:,d}")
print(f"INNER JOIN antes:               {total_inner:,d} (perda: {perda:,d} / {pct_perda:.2f}%)")
print(f"INNER JOIN agora:               {total_inner_novo:,d} (perda: {perda_nova:,d} / {pct_perda_nova:.2f}%)")
print(f"Registros recuperados:          {total_inner_novo - total_inner:,d}")

## Incorporar aeródromos faltantes do SIROS

O SIROS contém todos os 28 códigos OACI ausentes na nossa dimensão `silver_aerodromos`. Vamos filtrar apenas esses registros, adaptar o schema e inseri-los na dimensão.

**Mapeamento de colunas SIROS → silver_aerodromos:**

| SIROS | silver_aerodromos |
| --- | --- |
| SIGLA ICAO AERÓDROMO | codigo_oaci |
| NOME AERÓDROMO | nome |
| MUNICÍPIO AERÓDROMO | municipio / municipio_servido |
| ESTADO AERÓDROMO | uf / uf_servido |
| LATITUDE | latgeopoint |
| LONGITUDE | longeopoint |
| — | latitude, longitude (DMS), altitude, ciad, situacao = nulos |
| NOME + OACI | aerodromo (formato: `NOME (OACI)`) |

In [0]:
from pyspark.sql.functions import col, concat, lit, upper, trim, regexp_replace

missing_codes = list(set(
    [r[0] for r in aero_orig_keys_miss.select("sigla_icao_aeroporto_origem").collect()] +
    [r[0] for r in aero_dest_keys_miss.select("sigla_icao_aeroporto_destino").collect()]
))

aeros_faltantes = aero_siros.filter(
    upper(trim(col("SIGLA ICAO AERÓDROMO"))).isin(missing_codes)
)

print(f"Aeródromos faltantes encontrados no SIROS: {aeros_faltantes.count()}")
display(aeros_faltantes.select("SIGLA ICAO AERÓDROMO", "NOME AERÓDROMO", "MUNICÍPIO AERÓDROMO", "ESTADO AERÓDROMO", "PAÍS AERÓDROMO"))

In [0]:
aeros_faltantes_fmt = (aeros_faltantes
    .select(
        upper(trim(col("SIGLA ICAO AERÓDROMO"))).alias("codigo_oaci"),
        lit(None).cast("string").alias("ciad"),
        upper(trim(col("NOME AERÓDROMO"))).alias("nome"),
        upper(trim(col("MUNICÍPIO AERÓDROMO"))).alias("municipio"),
        upper(trim(col("ESTADO AERÓDROMO"))).alias("uf"),
        upper(trim(col("MUNICÍPIO AERÓDROMO"))).alias("municipio_servido"),
        upper(trim(col("ESTADO AERÓDROMO"))).alias("uf_servido"),
        regexp_replace(col("LATITUDE"), ",", ".").cast("double").alias("latgeopoint"),
        regexp_replace(col("LONGITUDE"), ",", ".").cast("double").alias("longeopoint"),
        lit(None).cast("string").alias("latitude"),
        lit(None).cast("string").alias("longitude"),
        lit(None).cast("double").alias("altitude"),
        lit("CADASTRADO").alias("situacao"),
    )
    .withColumn("aerodromo", concat(col("nome"), lit(" ("), col("codigo_oaci"), lit(")")))
)

colunas_silver = [c.name for c in spark.table("mvp_engenharia_dados.silver.silver_aerodromos").schema.fields]
aeros_faltantes_fmt = aeros_faltantes_fmt.select(*colunas_silver)

aerodromos_atualizado = aerodromos.unionByName(aeros_faltantes_fmt)

(aerodromos_atualizado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"silver_aerodromos atualizada: {aerodromos_atualizado.count()} aeródromos (antes: {aerodromos.count()})")
display(spark.table("mvp_engenharia_dados.silver.silver_aerodromos")
    .filter(col("codigo_oaci").isin(missing_codes))
    .select("codigo_oaci", "nome", "municipio", "uf", "aerodromo"))

### Re-executar verificacão de impacto final

Com ambas as dimensões atualizadas (empresas + aeródromos), re-executamos o INNER JOIN para medir a taxa de perda final.

In [0]:
aerodromos_novo = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

vra_completo_final = (vra
    .join(empresas_nova, vra.sigla_icao_empresa_aerea == empresas_nova.icao, "inner")
    .join(aerodromos_novo, vra.sigla_icao_aeroporto_origem == aerodromos_novo.codigo_oaci, "inner")
    .join(aerodromos_novo.alias("aero_dest"), vra.sigla_icao_aeroporto_destino == col("aero_dest.codigo_oaci"), "inner"))

total_inner_final = vra_completo_final.count()
perda_final = total_vra - total_inner_final
pct_perda_final = perda_final / total_vra * 100

print("=" * 60)
print("VERIFICACÃO FINAL (empresas + aeródromos atualizados)")
print("=" * 60)
print(f"Registros originais (VRA):      {total_vra:,d}")
print(f"INNER JOIN inicial:            {total_inner:,d} (perda: {perda:,d} / {pct_perda:.2f}%)")
print(f"INNER JOIN pós empresas:       {total_inner_novo:,d} (perda: {perda_nova:,d} / {pct_perda_nova:.2f}%)")
print(f"INNER JOIN final:              {total_inner_final:,d} (perda: {perda_final:,d} / {pct_perda_final:.2f}%)")
print(f"\nTotal de registros recuperados: {total_inner_final - total_inner:,d}")
print(f"Perda remanescente:            {perda_final:,d} ({pct_perda_final:.2f}%)")

## Verificação de voos com aeroportos estrangeiros

Alguns dos aeródromos incorporados do SIROS são claramente estrangeiros (SAEZ, SCEL, SKBO, etc.). No entanto, a tabela `silver_vra` foi filtrada na camada silver para conter apenas voos domésticos (`Código Tipo Linha = N`). Vamos investigar por que esses voos aparecem.

In [0]:
aero_siros_br = aero_siros.filter(upper(trim(col("PAÍS AERÓDROMO"))) == "BRASIL")
codigos_br = set([r[0] for r in aero_siros_br.select(upper(trim(col("SIGLA ICAO AERÓDROMO")))).distinct().collect()])

vra_orig_estrangeira = vra.filter(~col("sigla_icao_aeroporto_origem").isin(codigos_br))
vra_dest_estrangeiro = vra.filter(~col("sigla_icao_aeroporto_destino").isin(codigos_br))

print(f"Voos com ORIGEM estrangeira: {vra_orig_estrangeira.count()}")
print(f"Voos com DESTINO estrangeiro: {vra_dest_estrangeiro.count()}")
print(f"Voos com ambos estrangeiros: {vra.filter(~col('sigla_icao_aeroporto_origem').isin(codigos_br) & ~col('sigla_icao_aeroporto_destino').isin(codigos_br)).count()}")
print()
print("Top 10 origens estrangeiras:")
display(vra_orig_estrangeira.groupBy("sigla_icao_aeroporto_origem").count().orderBy(col("count").desc()).limit(10))
print("\nTop 10 destinos estrangeiros:")
display(vra_dest_estrangeiro.groupBy("sigla_icao_aeroporto_destino").count().orderBy(col("count").desc()).limit(10))

### Inspeção dos voos estrangeiros

Vamos examinar amostras desses voos para entender por que estão classificados como domésticos. O filtro original usou `Código Tipo Linha = N` (Doméstica Mista) na bronze, mas é possível que:
1. O campo `Código Tipo Linha` tenha sido preenchido incorretamente pela empresa aérea
2. A definição de "doméstica" pela ANAC permita etapas internacionais em rotas domésticas
3. Exista erro de digitação no código ICAO do aeroporto

In [0]:
from pyspark.sql.functions import concat, lit

vra_estrangeiro = vra.filter(
    ~col("sigla_icao_aeroporto_origem").isin(codigos_br) |
    ~col("sigla_icao_aeroporto_destino").isin(codigos_br)
)

display(vra_estrangeiro
    .withColumn("rota", concat(col("sigla_icao_aeroporto_origem"), lit(" → "), col("sigla_icao_aeroporto_destino")))
    .groupBy("sigla_icao_empresa_aerea", "rota")
    .count()
    .orderBy(col("count").desc())
    .limit(20))

In [0]:
display(vra_estrangeiro
    .select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "sigla_icao_aeroporto_origem",
        "sigla_icao_aeroporto_destino",
        "partida_prevista",
        "chegada_prevista",
        "descricao_etapa",
        "referencia",
    )
    .orderBy(col("sigla_icao_empresa_aerea"), col("partida_prevista"))
    .limit(25))

## Filtrar voos com aeroportos estrangeiros

Como a análise é estritamente doméstica, os 44 registros com aeroportos estrangeiros (erro de classificação na origem ANAC) serão removidos da `silver_vra`. O critério é manter apenas registros onde origem E destino são aeródromos brasileiros segundo o SIROS.

In [0]:
vra_filtrado = vra.filter(
    col("sigla_icao_aeroporto_origem").isin(codigos_br) &
    col("sigla_icao_aeroporto_destino").isin(codigos_br)
)

removidos = total_vra - vra_filtrado.count()

(vra_filtrado.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_vra"))

print(f"Registros originais: {total_vra:,d}")
print(f"Registros removidos: {removidos:,d} (voos com aeroportos estrangeiros)")
print(f"silver_vra atualizada: {vra_filtrado.count():,d} registros")

### Verificação final após filtro

Re-executamos o INNER JOIN com as dimensões atualizadas sobre a VRA filtrada para confirmar que a perda residual é mínima.

In [0]:
vra_final = spark.table("mvp_engenharia_dados.silver.silver_vra")
total_vra_final = vra_final.count()

vra_completo_final = (vra_final
    .join(empresas_nova, vra_final.sigla_icao_empresa_aerea == empresas_nova.icao, "inner")
    .join(aerodromos_novo, vra_final.sigla_icao_aeroporto_origem == aerodromos_novo.codigo_oaci, "inner")
    .join(aerodromos_novo.alias("aero_dest"), vra_final.sigla_icao_aeroporto_destino == col("aero_dest.codigo_oaci"), "inner"))

total_inner_final = vra_completo_final.count()
perda_final = total_vra_final - total_inner_final
pct_perda_final = perda_final / total_vra_final * 100

print("=" * 60)
print("VERIFICAÇÃO FINAL (VRA filtrada + dimensões atualizadas)")
print("=" * 60)
print(f"Registros VRA (após filtro):   {total_vra_final:,d}")
print(f"INNER JOIN final:              {total_inner_final:,d}")
print(f"Registros perdidos:           {perda_final:,d} ({pct_perda_final:.2f}%)")
print(f"Registros mantidos:           {total_inner_final / total_vra_final * 100:.4f}%")

## Remover o registro ARN

O voo ARN 0767 (SBPA → SBGR, fretamento) é o único registro com empresa inexistente. O código ARN não consta no SIROS, nem na [List of airline codes](https://en.wikipedia.org/wiki/List_of_airline_codes) da Wikipedia — provavelmente erro de digitação na origem ANAC. Vamos removê-lo para zerar a perda no INNER JOIN.

In [0]:
vra_sem_arn = vra_final.filter(col("sigla_icao_empresa_aerea") != "ARN")

(vra_sem_arn.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_vra"))

print(f'silver_vra final: {vra_sem_arn.count():,d} registros (removido 1 registro ARN)')

In [0]:
vra_final_ok = spark.table("mvp_engenharia_dados.silver.silver_vra")
empresas_ok = spark.table("mvp_engenharia_dados.silver.silver_empresas")
aerodromos_ok = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

total = vra_final_ok.count()
inner_join = (vra_final_ok
    .join(empresas_ok, vra_final_ok.sigla_icao_empresa_aerea == empresas_ok.icao, "inner")
    .join(aerodromos_ok, vra_final_ok.sigla_icao_aeroporto_origem == aerodromos_ok.codigo_oaci, "inner")
    .join(aerodromos_ok.alias("d"), vra_final_ok.sigla_icao_aeroporto_destino == col("d.codigo_oaci"), "inner")
).count()

perda = total - inner_join
pct = perda / total * 100
print(f'Registros VRA: {total:,d}')
print(f'INNER JOIN:    {inner_join:,d}')
print(f'Perda:         {perda} ({pct:.4f}%)')
print(f'Cobertura:     {inner_join / total * 100:.4f}%')

## Remover aeródromos estrangeiros da dimensão

Como a análise é estritamente doméstica e os voos estrangeiros já foram filtrados da `silver_vra`, os aeródromos estrangeiros incorporados do SIROS não são mais necessários. Vamos manter apenas aeródromos com `uf` preenchido (brasileiros).

In [0]:
aerodromos_br = aerodromos_ok.filter(col("uf").isNotNull())

removidos_aero = aerodromos_ok.count() - aerodromos_br.count()

(aerodromos_br.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f'Antes: {aerodromos_ok.count()} aeródromos')
print(f'Removidos: {removidos_aero} aeródromos estrangeiros')
print(f'Agora: {aerodromos_br.count()} aeródromos')

In [0]:
aero_final = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")
emp_final = spark.table("mvp_engenharia_dados.silver.silver_empresas")
vra_final_ok = spark.table("mvp_engenharia_dados.silver.silver_vra")

total = vra_final_ok.count()
inner_join = (vra_final_ok
    .join(emp_final, vra_final_ok.sigla_icao_empresa_aerea == emp_final.icao, "inner")
    .join(aero_final, vra_final_ok.sigla_icao_aeroporto_origem == aero_final.codigo_oaci, "inner")
    .join(aero_final.alias("d"), vra_final_ok.sigla_icao_aeroporto_destino == col("d.codigo_oaci"), "inner")
).count()

perda = total - inner_join
pct = perda / total * 100
print(f'Registros VRA: {total:,d}')
print(f'INNER JOIN:    {inner_join:,d}')
print(f'Perda:         {perda} ({pct:.4f}%)')
print(f'Cobertura:     {inner_join / total * 100:.4f}%')

## Resumo dos tratamentos aplicados

### Estado inicial

Partindo das tabelas silver criadas no notebook `04-camada-silver`, a verificacão de integridade referencial revelou:

| Métrica | Valor |
| --- | --- |
| Registros na VRA | 2.736.859 |
| Empresas na dimensão | 19 |
| Aeródromos na dimensão | 496 |
| Perda no INNER JOIN | 47.843 (1,75%) |

### Tratamentos aplicados

**1. Complementação de empresas (PTB e PAM)**

- Baixamos o CSV de companhias aéreas do SIROS/ANAC para o volume de staging
- O SIROS não continha os códigos PTB, PAM e ARN
- Identificamos PTB = Voepass Linhas Aéreas e PAM = MAP Linhas Aéreas (fonte: Wikipedia)
- Inserimos manualmente as duas empresas na dimensão `silver_empresas`
- **Recuperados: 38.777 registros**

**2. Complementação de aeródromos (SIROS)**

- Baixamos o CSV de aeródromos do SIROS (10.560 aeródromos mundiais)
- Todos os 28 códigos OACI ausentes foram encontrados no SIROS
- Adaptamos o schema (mapeamento de colunas, campos nulos para ciad/altitude/DMS)
- Incorporamos os 28 aeródromos na dimensão `silver_aerodromos`
- **Recuperados: 11.840 registros**

**3. Remoção de voos com aeroportos estrangeiros**

- Identificamos 44 voos classificados como domésticos (Código Tipo Linha = N) mas com origem ou destino em aeroportos estrangeiros
- Causa: erro de classificação na origem ANAC — a maioria eram voos charter/fretamento da Sideral (SID) e Gol (GLO)
- Como a análise é estritamente doméstica, removemos esses 44 registros da `silver_vra`

**4. Remoção do código ARN**

- O código ARN (1 registro na VRA) não existe no SIROS nem na [List of airline codes](https://en.wikipedia.org/wiki/List_of_airline_codes) da Wikipedia
- Provável erro de digitação na base ANAC (ARN é código IATA do aeroporto de Estocolmo-Arlanda)
- Removido o registro da `silver_vra`

**5. Limpeza de aeródromos estrangeiros da dimensão**

- Após filtrar os voos estrangeiros da VRA, os 20 aeródromos estrangeiros incorporados do SIROS não eram mais utilizados
- Removidos da dimensão `silver_aerodromos`, mantendo apenas aeródromos brasileiros (com `uf` preenchido)

### Estado final

| Tabela | Antes | Depois |
| --- | --- | --- |
| `silver_vra` | 2.736.859 | 2.736.814 |
| `silver_empresas` | 19 | 21 |
| `silver_aerodromos` | 496 | 504 |
| **Cobertura INNER JOIN** | **98,25%** | **100,00%** |